In [1]:
from rldb.utils import *

In [2]:
repo_id = "rpuns/test"
episodes = [0]
dataset_path = '/home/rl2-eve/EgoVerse/logs/lerobot/lerobot'
model_path = '/home/rl2-eve/EgoVerse/logs/cotrain_scoop_granular_everse/cartesian_sep_2025-07-05_16-14-08/0/everse_eval_test/eval_test_cartesian_sep_2025-07-05_16-14-08/checkpoints/epoch=999-step=100000.ckpt'

In [3]:
dataset = RLDBDataset(repo_id=repo_id, root=dataset_path, local_files_only=True, episodes=episodes, mode="sample")

Returning existing local_dir `/home/rl2-eve/EgoVerse/logs/lerobot/lerobot` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/home/rl2-eve/EgoVerse/logs/lerobot/lerobot` as remote repo cannot be accessed in `snapshot_download` (None).
Returning existing local_dir `/home/rl2-eve/EgoVerse/logs/lerobot/lerobot` as remote repo cannot be accessed in `snapshot_download` (None).


Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
eval_model = None
from typing import Any, Dict, List, Optional, Tuple

import os
import copy

import hydra
import lightning as L
from lightning import Callback, LightningDataModule, LightningModule, Trainer

from hydra import compose, initialize

from omegaconf import DictConfig, OmegaConf

from egomimic.utils.pylogger import RankedLogger
from egomimic.utils.utils import extras, task_wrapper

from egomimic.pl_utils.pl_model import ModelWrapper

from egomimic.scripts.evaluation.eval import Eval

log = RankedLogger(__name__, rank_zero_only=True)

from rldb.utils import DataSchematic

@task_wrapper
def eval(cfg: DictConfig):
    if cfg.get("seed"):
        L.seed_everything(cfg.seed, workers=True)
    
    log.info(f"Instantiating data schematic <{cfg.multirun_cfg.data_schematic._target_}>")
    data_schematic: DataSchematic = hydra.utils.instantiate(cfg.multirun_cfg.data_schematic)
    datamodule = None
    if cfg.datasets is not None:
        
        if cfg.datasets == "multirun":
            log.info(f"Using multirun validation datasets")
            eval_datasets = cfg.multirun_cfg.data.valid_datasets
            datasets_target = cfg.multirun_cfg.data._target_
        elif "eval_datasets" in cfg.datasets and cfg.datasets.eval_datasets is not None:
            log.info(f"Using specified yaml evaluation datasets")
            eval_datasets = cfg.datasets.data.eval_datasets
            datasets_target = cfg.datasets.data._target_
        elif "valid_datasets" in cfg.datasets and cfg.datasets.valid_datasets is not None:
            log.ingo(f"Using specified yaml validation datasets")
            eval_datasets = cfg.datasets.data.valid_datasets
            datasets_target = cfg.datasets.data._target_
        
        eval_datasets_dict = {}
        for dataset_name in eval_datasets:
            eval_datasets[dataset_name] = hydra.utils.instantiate(
                eval_datasets_dict[dataset_name]
            )
    
        log.info(f"Instantiating datamodule <{datasets_target}>")
        assert "MultiDataModuleWrapper" in datasets_target, "cfg.data._target_ must be 'MultiDataModuleWrapper'"
        datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data, valid_datasets=eval_datasets_dict)
    
        for dataset_name, dataset in datamodule.valid_datasets.items():
            log.info(f"Inferring shapes for dataset <{dataset_name}>")
            data_schematic.infer_shapes_from_batch(dataset[0])
            data_schematic.infer_norm_from_dataset(dataset)
    
    eval = hydra.utils.instantiate(cfg.eval)
    eval.datamodule = datamodule
    eval.data_schematic = data_schematic # unsure if this is necessary to pass in
    eval_model = eval

def build_cfg(
  config_path: str = "./hydra_configs",
  config_name: str = "eval.yaml",
  overrides: list[str] | None = None,
) -> OmegaConf:
  overrides = overrides or []
  with initialize(version_base="1.3", config_path=config_path):
    cfg = compose(config_name=config_name, overrides=overrides)

  # handle nested multirun config (your original logic)
  if "multirun_cfg" in cfg and isinstance(cfg.multirun_cfg, str):
    multi_cfg = OmegaConf.load(cfg.multirun_cfg)
    OmegaConf.set_struct(cfg, False)
    cfg["multirun_cfg"] = copy.deepcopy(multi_cfg)
    OmegaConf.set_struct(cfg, True)

  extras(cfg)          # same utility call you had in main()
  return cfg

pybullet build time: Jan 29 2025 23:16:28


In [5]:
cfg = build_cfg(
  config_path="../../hydra_configs",
  config_name="eval.yaml",
)

/home/rl2-eve/EgoVerse/.venv/lib/python3.10/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'eval.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
[rank: 0] Extras config not found! <cfg.extras=null>


In [8]:
type(cfg)

omegaconf.dictconfig.DictConfig

In [9]:
eval(cfg)

[rank: 0] 
Traceback (most recent call last):
  File "/home/rl2-eve/EgoVerse/.venv/lib/python3.10/site-packages/hydra/_internal/instantiate/_instantiate2.py", line 92, in _call_target
    return _target_(*args, **kwargs)
  File "/home/rl2-eve/EgoVerse/egomimic/scripts/evaluation/eve_eval.py", line 81, in __init__
    breakpoint()
  File "/home/rl2-eve/EgoVerse/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/model_helpers.py", line 125, in wrapper
    return self.method(cls, *args, **kwargs)
  File "/home/rl2-eve/EgoVerse/.venv/lib/python3.10/site-packages/lightning/pytorch/core/module.py", line 1611, in load_from_checkpoint
    loaded = _load_from_checkpoint(
  File "/home/rl2-eve/EgoVerse/.venv/lib/python3.10/site-packages/lightning/pytorch/core/saving.py", line 91, in _load_from_checkpoint
    model = _load_state(cls, checkpoint, strict=strict, **kwargs)
  File "/home/rl2-eve/EgoVerse/.venv/lib/python3.10/site-packages/lightning/pytorch/core/saving.py", line 165, in _l

ConfigAttributeError: Key 'paths' is not in struct
    full_key: paths
    object_type=dict

In [ ]:
eval_model